In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.transforms as mtransforms
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

# =============================================================================
# 1. Publication-Quality Visualization Settings (Nature Portfolio Standard)
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
DPI_SETTING = 600

# Force MathText mapping for Bold and Bold-Italic in PDF output
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial:italic:bold'
plt.rcParams['mathtext.bf'] = 'Arial:bold'

# =============================================================================
# 2. Robust Data Loading & Strict Exclusion Rules
# =============================================================================
def clean_and_convert_to_nan(df, cols, is_qpcr=False):
    for col in cols:
        s = df[col].astype(str).replace(['Undetermined', '-', 'nan', 'NaN', '#VALUE!', ''], np.nan)
        if is_qpcr:
            s = s.str.replace(r'\.E\+', 'E+', regex=True)
            s = s.str.replace(r'\.E\-', 'E-', regex=True)
        df[col] = pd.to_numeric(s, errors='coerce')
    return df

# Load datasets
files = {
    'pH': 'pH.csv',
    'Butyrate': 'Butyrate(mM).csv',
    'Acetate': 'Acetate(mM).csv',
    'Bifidobacterium': 'Bifidobacterium(qPCR).csv',
    'Faecalibacterium': 'Faecalibacterium(qPCR).csv'
}

dfs = {k: pd.read_csv(v) for k, v in files.items()}
donor_cols = [c for c in dfs['pH'].columns if c.startswith('HS-')]

# Apply strict cleaning
for k in ['pH', 'Butyrate', 'Acetate']:
    dfs[k] = clean_and_convert_to_nan(dfs[k], donor_cols)
for k in ['Bifidobacterium', 'Faecalibacterium']:
    dfs[k] = clean_and_convert_to_nan(dfs[k], donor_cols, is_qpcr=True)

# =============================================================================
# 3. Calculate Delta (Inulin - Control)
# =============================================================================
data = {'Donor': donor_cols}

for k in files.keys():
    ctrl = dfs[k][dfs[k]['KULFFI'].str.strip() == 'Control'][donor_cols].iloc[0]
    inulin = dfs[k][dfs[k]['KULFFI'].str.strip() == 'Inulin'][donor_cols].iloc[0]

    if k in ['Bifidobacterium', 'Faecalibacterium']:
        data[f'Delta_{k}'] = np.log10(inulin + 1) - np.log10(ctrl + 1)
    else:
        data[f'Delta_{k}'] = inulin - ctrl

df_all = pd.DataFrame(data)

# =============================================================================
# 4. Ecological Stratification (Based on Universal pH Tipping Points)
# =============================================================================
def classify_ecotype(val):
    if pd.isna(val):
        return np.nan
    if val <= -0.80:
        return 'Severe'
    elif -0.80 < val < -0.44:
        return 'Moderate'
    else:
        return 'Mild'

df_all['Ecotype'] = df_all['Delta_pH'].apply(classify_ecotype)
ecotype_order = ['Severe', 'Moderate', 'Mild']
palette = {'Mild': '#d62728', 'Moderate': '#7f7f7f', 'Severe': '#1f77b4'}

# =============================================================================
# 5. Statistical FDR Annotation Function (Strict q-values)
# =============================================================================
def add_fdr_annotation(ax, data, val_col, group_col, order):
    pairs = [(order[0], order[1]), (order[1], order[2]), (order[0], order[2])]
    p_values = []

    for g1, g2 in pairs:
        d1 = data[data[group_col] == g1][val_col].dropna()
        d2 = data[data[group_col] == g2][val_col].dropna()
        if len(d1) > 0 and len(d2) > 0:
            stat, p = stats.mannwhitneyu(d1, d2, alternative='two-sided')
            p_values.append(p)
        else:
            p_values.append(1.0)

    # Benjamini-Hochberg FDR correction -> q_values
    _, q_values, _, _ = multipletests(p_values, method='fdr_bh')

    y_max = data[val_col].max()
    y_range = y_max - data[val_col].min()
    current_y = y_max + (y_range * 0.05)
    step = y_range * 0.12

    for (g1, g2), q in zip(pairs, q_values):
        if q < 0.05:
            sig_text = '***' if q < 0.001 else '**' if q < 0.01 else '*'
            x1, x2 = order.index(g1), order.index(g2)
            # Bracket (Black)
            ax.plot([x1, x1, x2, x2], [current_y, current_y+step*0.2, current_y+step*0.2, current_y], lw=1.2, c='black')
            # Asterisks (Dark Cyan: #008B8B)
            ax.text((x1+x2)*.5, current_y+step*0.2, sig_text, ha='center', va='bottom', color='#008B8B', fontsize=16, fontweight='bold')
            current_y += step

# =============================================================================
# 6. Plotting Figure 6 (4-Panel Mechanism Grid)
# =============================================================================
fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))

plot_configs = [
    {'col': 'Delta_Butyrate', 'title': r'$\mathbf{\Delta}$ Butyrate', 'ylabel': 'mM'},
    {'col': 'Delta_Acetate', 'title': r'$\mathbf{\Delta}$ Acetate', 'ylabel': 'mM'},
    {'col': 'Delta_Bifidobacterium', 'title': r'$\mathbf{\Delta}$ $\mathit{Bifidobacterium}$ spp.', 'ylabel': r'Log$_{\mathbf{10}}$ copies/mL'},
    {'col': 'Delta_Faecalibacterium', 'title': r'$\mathbf{\Delta}$ $\mathit{Faecalibacterium}$ spp.', 'ylabel': r'Log$_{\mathbf{10}}$ copies/mL'}
]

for idx, config in enumerate(plot_configs):
    ax = axes[idx]
    col = config['col']

    plot_data = df_all.dropna(subset=[col, 'Ecotype'])

    # Background Boxplot
    sns.boxplot(x='Ecotype', y=col, data=plot_data, order=ecotype_order,
                palette=palette, width=0.5, showfliers=False, ax=ax, boxprops=dict(alpha=0.6))

    # Foreground Stripplot
    sns.stripplot(x='Ecotype', y=col, data=plot_data, order=ecotype_order,
                  palette=palette, hue='Ecotype', legend=False,
                  alpha=0.9, size=6, jitter=True, edgecolor='black', linewidth=1.0, ax=ax, zorder=3)

    # Zero baseline
    ax.axhline(0, color='black', linestyle=':', linewidth=1.5, zorder=0)

    # Titles & Y-labels
    ax.set_title(config['title'], fontsize=16, fontweight='bold', pad=15)
    ax.set_ylabel(config['ylabel'], fontsize=14, fontweight='bold')
    ax.set_xlabel('')

    # X-axis positioning
    ax.set_xlim(-0.6, 2.6)
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(ecotype_order, fontsize=14, fontweight='bold', ha='center')
    ax.tick_params(axis='y', labelsize=12, width=1.5, length=6)

    # --- [CRITICAL UPDATE] Fine-tune X-tick label positions ---
    # Shift 'Moderate' right by half a character (~7 points) without moving the plots.
    # Severe and Mild remain exactly centered.
    labels = ax.xaxis.get_majorticklabels()
    shift_right = mtransforms.ScaledTranslation(7/72., 0, fig.dpi_scale_trans)

    labels[1].set_transform(labels[1].get_transform() + shift_right)  # Shift Moderate
    # ----------------------------------------------------------

    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(2.0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Add FDR annotations (q-values)
    add_fdr_annotation(ax, plot_data, col, 'Ecotype', ecotype_order)

# Adjust layout spacing to be perfectly uniform
plt.subplots_adjust(wspace=0.35)

# Export
output_file = 'Figure_6g-j.pdf'
plt.savefig(output_file, dpi=DPI_SETTING, bbox_inches='tight')
print(f"Figure successfully saved as {output_file}")